# 14. Dense Data Generation (main_V4)
**Objective:** Rebuild the Subject 1 dataset using a 10ms stride (95% overlap) to exponentially increase the sample size for SOTA Dictionary Learning classification.

In [1]:
%load_ext autoreload
%autoreload 2

import sys
import os
import h5py
import numpy as np

sys.path.append(os.path.abspath('../'))
from src.utils import load_ninapro_mat
from src.preprocess import sEMGPreprocessor, save_preprocessed_hdf5
from src.config import PREPROCESSED_DIR

### 1. Load Raw Data (Subject 1)

In [2]:
db_dir = "../data/raw/Ninapro_DB1/s1"
exercises = ['E1', 'E2', 'E3']
offsets = {'E1': 0, 'E2': 12, 'E3': 29} 

emg_list, labels_list, reps_list = [], [], []

for ex in exercises:
    mat_path = os.path.join(db_dir, f"S1_A1_{ex}.mat")
    data = load_ninapro_mat(mat_path)
    
    labels = data['labels'].copy()
    active_mask = labels > 0
    labels[active_mask] += offsets[ex]
    
    emg_list.append(data['emg'])
    labels_list.append(labels)
    reps_list.append(data['reps'])

emg_full = np.vstack(emg_list)
labels_full = np.concatenate(labels_list)
reps_full = np.concatenate(reps_list)

print(f"Raw Continuous Shape: {emg_full.shape}")

Raw Continuous Shape: (471483, 10)


### 2. Filter & Standardize

In [3]:
preprocessor = sEMGPreprocessor(database_name="DB1")
filtered_emg = preprocessor.filter_signal(emg_full)
norm_emg = preprocessor.standardize(filtered_emg)

### 3. Extract Dense Overlapping Windows (The V4 Upgrade)

In [4]:
# Using our new V4 extraction method
X_dense, y_dense, reps_dense = preprocessor.extract_dense_windows(norm_emg, labels_full, reps_full)

# Balance the Rest Class (Class 0). Because we have dense windows, the Rest class 
# will be massive. The existing balance_rest_class method works perfectly here.
print("\nBalancing Rest Class...")
X_bal, y_bal, reps_bal = preprocessor.balance_rest_class(X_dense, y_dense, reps_dense)

Extracting DENSE windows (stride=1 sample / 10ms)...
Dense extraction complete. Exploded dataset to 471464 windows (Shape: (471464, 20, 10)).

Balancing Rest Class...


### 4. Serialize the SOTA Dataset

In [5]:
dense_h5_path = os.path.join(PREPROCESSED_DIR, "DB1_subject_1_Dense.h5")
with h5py.File(dense_h5_path, 'w') as f:
    # Using compression to save disk space since the array is now massive
    f.create_dataset('X', data=X_bal, compression='gzip')
    f.create_dataset('y', data=y_bal)
    f.create_dataset('reps', data=reps_bal)

print(f"\nDense Dataset Serialized Successfully to: {dense_h5_path}")
print(f"Final V4 Tensor Shape: {X_bal.shape}")


Dense Dataset Serialized Successfully to: /workspaces/TCC/data/preprocessed/DB1_subject_1_Dense.h5
Final V4 Tensor Shape: (186753, 20, 10)
